# **Régimes du prix du pétrole et coût des intrants pétrochimiques**

## Étape 1 : construire la base de données de travail

Cet article reprend la méthode du GWP2 (modèle de Markov caché sur des variations standardisées) et l'applique à une question concrète : les coûts d'approvisionnement d'un manufacturier de revêtements de sol. Avant tout modèle, il faut une base de données propre, documentée et figée.

### 1.1 Séries de Statistique Canada

Nous partons du tableau 18100266 de Statistique Canada, l'indice des prix des produits industriels par produit, en base janvier 2020 égale à 100. Le fichier brut contient près de 400 produits mensuels, de 1956 à septembre 2026. Nous en gardons cinq, repérés par leur code NAPCS plutôt que par leur nom, ce qui évite les erreurs de correspondance :

* `[281]` résines plastiques (polypropylène, nylon en résine). Nous utilisons l'agrégat et non la série `[28111]`, qui s'arrête en juillet 2024 après le découpage de la classification.
* `[28221]` fibres et filaments synthétiques (nylon, polyester). Statistique Canada signale ces valeurs comme estimées.
* `[28211P]` caoutchouc synthétique (latex).
* `[26321]` produits pétrochimiques.
* `[23211]` tapis, moquettes et carpettes, qui représentent le prix du produit fini.

Ces séries sont ensuite fusionnées avec `oil_clean.csv`, le fichier nettoyé du GWP (WTI, variables macroéconomiques et financières). La fusion se fait sur les mois communs, ce qui ramène l'échantillon à janvier 2010 jusqu'à août 2026, soit 200 mois. C'est plus court que dans le GWP, parce que la plupart des séries de produits de Statistique Canada commencent en 2010. Nous le mentionnerons dans les limites de l'étude.

Fichiers utilisés et produits à cette étape :

* `statcan_ippi_brut.csv` : tableau complet téléchargé (ancien nom `18100266.csv`).
* `statcan_ippi_metadata.csv` : documentation du tableau (ancien nom `18100266_MetaData.csv`).
* `oil_clean.csv` : base du GWP, inchangée.
* `intrants_oil_merged.csv` : résultat de la fusion.

In [6]:
#!pip install hmmlearn

In [1]:
import os, pandas as pd
#os.rename("18100266.csv", "statcan_ippi_brut.csv")
#os.rename("18100266_MetaData.csv", "statcan_ippi_metadata.csv")
s = pd.read_csv("statcan_ippi_brut.csv", usecols=["REF_DATE","North American Product Classification System (NAPCS)","VALUE"])
s.columns = ["date","produit","valeur"]
codes = {"[281]":"Resines plastiques","[28221]":"Fibres synthetiques","[28211P]":"Caoutchouc synthetique","[26321]":"Petrochimiques","[23211]":"Tapis"}
s["code"] = s.produit.str.extract(r"(\[[^\]]+\])\s*$")[0]
x = s[s.code.isin(codes)].pivot(index="date", columns="code", values="valeur").rename(columns=codes)
x.index = pd.to_datetime(x.index + "-01")
oil = pd.read_csv("oil_clean.csv", index_col="DATE", parse_dates=True)
data = oil.join(x, how="inner").dropna()
data.to_csv("intrants_oil_merged.csv")
print(data.shape, data.index[0].date(), data.index[-1].date())

(200, 20) 2010-01-01 2026-08-01


### 1.2 Énergie, change et indices américains

La question du manufacturier ne se résume pas au pétrole en dollars américains. Nous ajoutons donc quatre séries de la Réserve fédérale de St. Louis (FRED), pour deux raisons : compléter le portrait des coûts, et tester la robustesse des résultats.

* Gaz naturel Henry Hub (`MHHNGSP`), matière première de la pétrochimie nord américaine, dont le prix peut bouger indépendamment du pétrole.
* Taux de change CAD par USD (`DEXCAUS`). Il permet de calculer le WTI exprimé en dollars canadiens, soit le coût du pétrole tel que le vit un acheteur canadien.
* Indices américains des résines (`PCU325211325211`) et du caoutchouc synthétique (`PCU325212325212`) du Bureau of Labor Statistics. Si les mêmes régimes apparaissent avec les indices canadiens et américains, le résultat est plus solide.

Nous nous limitons volontairement à ces quatre ajouts : avec environ 200 observations mensuelles, chaque variable supplémentaire alourdit le modèle et réduit la fiabilité de l'estimation. Les valeurs manquantes en fin de période, dues au délai de publication, sont comptées puis retirées, sans remplissage.

La base finale est enregistrée sous `intrants_oil_complet.csv`. À partir d'ici, c'est ce fichier, et lui seul, qui alimente toutes les analyses de régimes. Les règles de nettoyage sont fixées avant d'observer le moindre résultat du modèle.

In [3]:
from pandas_datareader import data as pdr
ids = {"MHHNGSP":"Gaz naturel","DEXCAUS":"CAD par USD","PCU325211325211":"BLS resines","PCU325212325212":"BLS caoutchouc"}
f = pdr.DataReader(list(ids), "fred", "2010/01/01").resample("MS").mean().rename(columns=ids)
data = pd.read_csv("intrants_oil_merged.csv", index_col=0, parse_dates=True).join(f, how="left")
data["WTI en CAD"] = data["WTI spot"] * data["CAD par USD"]
data.index.name = "DATE"
print(data.isna().sum()[lambda s: s > 0])
data.dropna().to_csv("intrants_oil_complet.csv")

Series([], dtype: int64)


Là on vérifie si les variables sont complètes

In [4]:
d = pd.read_csv("intrants_oil_complet.csv", index_col="DATE", parse_dates=True)
print(d.shape, d.index[0].date(), d.index[-1].date())

(200, 25) 2010-01-01 2026-08-01


Aucune valeur manquante n'a été observée après la fusion : les quatre séries couvrent la totalité des 200 mois de l'échantillon, de janvier 2010 à août 2026, et aucune interpolation n'a été nécessaire.

## Étape 2 : détection des régimes

Chaque série est transformée en variations logarithmiques mensuelles puis standardisée, car un modèle de régimes ne peut pas s'ajuster sur des niveaux de prix qui montent avec le temps. Un modèle de Markov caché à émissions normales est ensuite ajusté séparément sur chaque série, avec cinq points de départ aléatoires pour éviter un maximum local.

Le nombre d'états n'est pas fixé d'avance : nous comparons deux, trois et quatre états par le critère d'information bayésien (BIC) et retenons le plus bas. Les états sont classés par volatilité croissante et non par moyenne, car un classement par moyenne peut étiqueter « stagnant » un état qui est en réalité un état de choc.

In [7]:
import numpy as np, warnings; warnings.filterwarnings("ignore")
from hmmlearn.hmm import GaussianHMM
data = pd.read_csv("intrants_oil_complet.csv", index_col="DATE", parse_dates=True)
cols = ["WTI en CAD","Resines plastiques","Fibres synthetiques","Caoutchouc synthetique","Petrochimiques","Tapis","Gaz naturel"]
X = np.log(data[cols]).diff().dropna()
Z = (X - X.mean()) / X.std()
def regimes(s, tries=5):
    z = s.to_numpy().reshape(-1, 1); best = None
    for k in (2, 3, 4):
        m = max((GaussianHMM(k, n_iter=300, random_state=r).fit(z) for r in range(tries)), key=lambda m: m.score(z))
        bic = -2 * m.score(z) + (k*k + 2*k - 1) * np.log(len(z))
        if best is None or bic < best[0]: best = (bic, k, m)
    _, k, m = best
    rank = {o: n for n, o in enumerate(np.argsort(m.covars_.ravel()))}
    return pd.Series(m.predict(z), index=s.index).map(rank), k
R, K = {}, {}
for c in cols: R[c], K[c] = regimes(Z[c])
R = pd.DataFrame(R); R.to_csv("regimes.csv")
for c in cols: print(c, "| etats:", K[c], "| parts:", R[c].value_counts(normalize=True).sort_index().round(2).tolist())

WTI en CAD | etats: 2 | parts: [0.97, 0.03]
Resines plastiques | etats: 2 | parts: [0.6, 0.4]
Fibres synthetiques | etats: 2 | parts: [0.67, 0.33]
Caoutchouc synthetique | etats: 2 | parts: [0.92, 0.08]
Petrochimiques | etats: 2 | parts: [0.92, 0.08]
Tapis | etats: 4 | parts: [0.29, 0.01, 0.58, 0.12]
Gaz naturel | etats: 2 | parts: [0.93, 0.07]


## Étape 3 : le pétrole annonce t il les chocs sur les intrants ?

Les régimes de l'étape précédente décrivent chaque série séparément. Il reste à tester l'idée centrale de l'article : si le pétrole entre en régime de choc, les coûts des intrants suivent ils quelques mois plus tard ?

### Méthode
Le WTI exprimé en dollars canadiens ne passe en état de choc que pendant cinq mois sur 199, regroupés en deux ou trois épisodes. Chaque pourcentage ci dessus repose donc sur très peu d'observations, un seul mois représentant 20 % : ces résultats sont indicatifs et ne constituent pas une preuve statistique. Pour cette raison, le lien entre le pétrole et les intrants est ensuite évalué sur l'ensemble des 199 mois à l'aide de corrélations décalées.


### Séries exclues

* **Tapis, moquettes et carpettes** : le prix chute de 20 % en novembre 2020 puis reste presque stable, avec 32 % des mois sans aucune variation. Ce saut isolé ressemble à un ajustement de l'indice et non à un choc de marché, et il fausse la détection des régimes. Cette série est conservée à titre de référence pour les graphiques, mais elle n'entre pas dans le test.
* **Résines plastiques** : leur état de forte volatilité forme un seul bloc continu de février 2020 à août 2026. Il s'agit d'un changement durable du niveau d'instabilité et non d'épisodes de choc. Un test de décalage n'a donc pas de sens pour cette série, nous la traitons à part dans la discussion.

### Limites

Le WTI ne passe en état de choc que pendant environ sept mois sur 199. Chaque pourcentage ci dessous repose donc sur très peu d'observations : il indique une tendance et ne constitue pas une preuve statistique. Nous présentons ces résultats comme exploratoires.

In [8]:
w = R["WTI en CAD"] == 1
print("mois de choc du WTI :", int(w.sum()), "sur", len(w))
for c in ["Petrochimiques","Fibres synthetiques","Caoutchouc synthetique","Gaz naturel"]:
    print(c, [round(((R[c].shift(-L) == 1) & w).sum() / w.sum(), 2) for L in range(5)], "base", round((R[c] == 1).mean(), 2))

mois de choc du WTI : 5 sur 199
Petrochimiques [np.float64(0.2), np.float64(0.2), np.float64(0.0), np.float64(0.0), np.float64(0.0)] base 0.08
Fibres synthetiques [np.float64(0.8), np.float64(0.6), np.float64(0.4), np.float64(0.2), np.float64(0.0)] base 0.33
Caoutchouc synthetique [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)] base 0.08
Gaz naturel [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)] base 0.07


## Étape 4 : résultats, le décalage entre le pétrole et les intrants

Le test par régimes de l'étape précédente repose sur seulement cinq mois de choc pétrolier, ce qui est insuffisant pour conclure. Nous mesurons donc le lien sur l'ensemble des 199 mois : pour chaque intrant, la corrélation entre sa variation mensuelle et celle du WTI en dollars canadiens observée L mois plus tôt, pour L de 0 à 6. Avec environ 198 observations, une corrélation n'est distinguable du hasard qu'au delà de 0,14 en valeur absolue.

### Lecture des résultats

* **Pétrochimiques** : lien fort, avec des corrélations de 0,46 le même mois et 0,50 un mois plus tard, qui s'éteignent ensuite.
* **Résines plastiques** : le pétrole se répercute surtout après un mois (0,43), avec un prolongement à deux mois (0,24).
* **Caoutchouc synthétique** : réaction plus lente et plus faible, avec un maximum de 0,23 à deux mois, qui reste positif jusqu'à quatre mois.
* **Fibres synthétiques et gaz naturel** : aucun lien net, les corrélations restent proches du seuil de signification.

Nous ne commentons pas les corrélations isolées à des décalages éloignés : sur 35 calculs, environ deux dépasseront le seuil par simple hasard.

### Portée et limites

Ces résultats décrivent des corrélations sur l'ensemble de la période et non des relations causales. Le WTI exprimé en dollars canadiens inclut l'effet du taux de change, que nous n'avons pas isolé. Enfin, la relation peut avoir changé dans le temps, notamment après 2020, ce que nous n'avons pas testé.

In [9]:
for c in ["Petrochimiques","Resines plastiques","Caoutchouc synthetique","Fibres synthetiques","Gaz naturel"]:
    print(c, [round(X[c].corr(X["WTI en CAD"].shift(L)), 2) for L in range(7)])

Petrochimiques [np.float64(0.46), np.float64(0.5), np.float64(0.23), np.float64(-0.0), np.float64(-0.14), np.float64(-0.04), np.float64(-0.02)]
Resines plastiques [np.float64(0.16), np.float64(0.43), np.float64(0.24), np.float64(0.04), np.float64(-0.11), np.float64(-0.24), np.float64(-0.11)]
Caoutchouc synthetique [np.float64(-0.08), np.float64(0.1), np.float64(0.23), np.float64(0.16), np.float64(0.15), np.float64(0.03), np.float64(-0.0)]
Fibres synthetiques [np.float64(-0.09), np.float64(-0.06), np.float64(0.13), np.float64(0.06), np.float64(0.03), np.float64(-0.05), np.float64(0.02)]
Gaz naturel [np.float64(0.05), np.float64(0.0), np.float64(0.2), np.float64(0.1), np.float64(-0.07), np.float64(-0.02), np.float64(0.05)]


### 4.1 Robustesse : la relation a t elle changé après 2020 ?

La pandémie et la poussée d'inflation de 2021 à 2022 ont profondément modifié les marchés de matières premières. Pour vérifier que les décalages précédents ne sont pas propres à une seule période, nous refaisons le calcul sur trois sous échantillons : avant 2020, depuis 2020, et depuis 2021 (ce dernier écarte les mois de la pandémie). Le seuil de signification est recalculé pour chaque période, car il dépend du nombre d'observations.

Une relation est considérée comme robuste si son décalage et son ordre de grandeur sont comparables d'une période à l'autre. Nous ne conservons dans la discussion que les relations qui satisfont ce critère.

In [10]:
cols = ["Petrochimiques","Resines plastiques","Caoutchouc synthetique","Fibres synthetiques"]
for nom, P in [("avant 2020", X[:"2019-12"]), ("depuis 2020", X["2020-01":]), ("depuis 2021", X["2021-01":])]:
    print(nom, len(P), "mois, seuil", round(1.96 / np.sqrt(len(P)), 2))
    for c in cols: print("  ", c, [round(P[c].corr(P["WTI en CAD"].shift(L)), 2) for L in range(5)])

avant 2020 119 mois, seuil 0.18
   Petrochimiques [np.float64(0.38), np.float64(0.5), np.float64(0.12), np.float64(-0.02), np.float64(-0.08)]
   Resines plastiques [np.float64(0.13), np.float64(0.31), np.float64(0.23), np.float64(0.2), np.float64(-0.03)]
   Caoutchouc synthetique [np.float64(-0.17), np.float64(0.1), np.float64(0.27), np.float64(0.19), np.float64(0.16)]
   Fibres synthetiques [np.float64(-0.28), np.float64(-0.04), np.float64(0.19), np.float64(-0.04), np.float64(0.08)]
depuis 2020 80 mois, seuil 0.22
   Petrochimiques [np.float64(0.53), np.float64(0.53), np.float64(0.31), np.float64(0.01), np.float64(-0.2)]
   Resines plastiques [np.float64(0.17), np.float64(0.48), np.float64(0.26), np.float64(0.02), np.float64(-0.14)]
   Caoutchouc synthetique [np.float64(-0.03), np.float64(0.1), np.float64(0.21), np.float64(0.15), np.float64(0.15)]
   Fibres synthetiques [np.float64(0.01), np.float64(-0.08), np.float64(0.1), np.float64(0.12), np.float64(-0.01)]
depuis 2021 68 mois, seu

### 4.2 Lecture de la robustesse

La relation entre le pétrole et les intrants est stable d'une période à l'autre, à une exception près.

* **Pétrochimiques** : le pic de corrélation se situe à un mois dans les trois périodes (0,50, 0,53 et 0,59). Le lien du même mois se renforce après 2020.
* **Résines plastiques** : le décalage d'un mois est constant, mais le lien se resserre nettement (0,31, 0,48 puis 0,66). Avant 2020, l'effet du pétrole s'étalait sur trois mois. Depuis 2020, il se concentre sur un à deux mois. La transmission est donc plus forte et plus rapide, ce qui est cohérent avec la hausse de volatilité des résines observée à l'étape 2, sans que nous puissions en établir la cause.
* **Caoutchouc synthétique** : le pic reste à deux mois (0,27, 0,21 et 0,25), mais son amplitude est proche du seuil de signification. Nous le retenons comme un lien modéré, à confirmer.
* **Fibres synthétiques** : aucune relation stable. La corrélation négative observée avant 2020 au décalage zéro disparaît ensuite et n'a pas d'interprétation économique. Statistique Canada qualifie ces valeurs d'estimées.

Les corrélations isolées à des décalages éloignés ne sont pas interprétées : sur 60 corrélations calculées, environ trois dépasseront le seuil de 5 % par simple hasard.

## **Références et sources des données**

### Sources de données
* **Statistique Canada.** Tableau 18-10-0266-01 : *Indice des prix des produits industriels, par produit, mensuel* (base janvier 2020 = 100), codes NAPCS `[281]` (résines plastiques), `[28221]` (fibres et filaments synthétiques), `[28211P]` (caoutchouc synthétique), `[26321]` (produits pétrochimiques) et `[23211]` (tapis, moquettes et carpettes).
* **Federal Reserve Bank of St. Louis (FRED).** Séries mensuelles : *Henry Hub Natural Gas Spot Price* (`MHHNGSP`) et *Canadian Dollars to U.S. Dollar Spot Exchange Rate* (`DEXCAUS`).
* **U.S. Bureau of Labor Statistics (BLS), via FRED.** *Producer Price Index by Industry: Plastics Material and Resin Manufacturing* (`PCU325211325211`) et *Synthetic Rubber Manufacturing* (`PCU325212325212`).
* **U.S. Energy Information Administration (EIA) / Base GWP.** Cours au comptant du pétrole brut *West Texas Intermediate* (`WTI spot`).

### Méthodologie et outils logiciels
* **Baum, L. E., & Petrie, T.** (1966). Statistical Inference for Probabilistic Functions of Finite State Markov Chains. *The Annals of Mathematical Statistics*, 37(6), 1554–1563.
* **Hamilton, J. D.** (1989). A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle. *Econometrica*, 57(2), 357–384.
* **Schwarz, G.** (1978). Estimating the Dimension of a Model. *The Annals of Statistics*, 6(2), 461–464.
* **Environnement technique :** Python 3, bibliothèques `pandas`, `numpy`, `pandas_datareader` et `hmmlearn` (`GaussianHMM`).